# Notebook 1: Dataset Cleaning and Website Availability Validation

This notebook prepares the company dataset for the scraping pipeline.

It cleans the raw company records, keeps only the target countries, removes unusable websites, maps company sizes into simple buckets, maps industries into broader sectors, checks company-name/domain match quality, separates suspicious duplicate domains for review, checks website availability, and saves the final CSV outputs.

The production output is `company_scrape_ready.csv`.

The testing output is `company_seed_sample.csv`.

## Pipeline summary

1. Import libraries and define project settings.
2. Clean column names and text values.
3. Keep rows with required fields.
4. Filter to the target countries.
5. Remove unusable website formats.
6. Extract normalised website URLs and domains.
7. Map company sizes into size buckets.
8. Map raw industries into high-level sectors.
9. Score company-name versus domain-name matches.
10. Keep strong domain matches.
11. Separate suspicious duplicate/shared domains into a review file.
12. Apply a first-pass country-domain signal.
13. Sample candidate companies by country, size, and sector.
14. Check website availability asynchronously.
15. Build the final scrape-ready dataset.
16. Explicitly enforce and validate the government organisation limit.
17. Save seed, production, review, reachable, and unreachable CSV outputs.

## Current limitations

These limitations are intentional and should be documented:

- The `51-200` size range is kept as `small` because the source dataset does not show exact employee counts. This is an approximate match to the 100+ staff requirement.
- `country_website_status` is only a domain suffix signal. It is not proof of headquarters or legal country of origin. Example: `.com.au` suggests Australia, but it does not prove that the company is Australian-owned or headquartered in Australia.
- Duplicate/shared domains are not automatically trusted. Suspicious cases are written to a review file and excluded from the automatic candidate pool.
- Domain matching is rule-based. Some brand names, abbreviations, transliterations, and non-Latin company names may still need manual review.
- Website availability only checks whether a website returns reachable HTML. It does not scrape company values yet.
- Robots.txt checking and values-page discovery should be handled in Notebook 2.

## 2. Imports and configuration

This block defines the full configuration for the notebook.

The most important project settings are:

- `TARGET_COUNTRIES`: countries kept in the dataset.
- `SIZE_CATEGORY_MAP`: raw employee-size ranges mapped into simple buckets.
- `MAX_FINAL_ROWS`: target final dataset size.
- `MAX_PER_COUNTRY`: target companies per country.
- `MAX_GOVERNMENT_RATIO`: maximum allowed government ratio.
- `MAX_GOVERNMENT_PER_COUNTRY`: max government organisations per country.
- `REQUEST_TIMEOUT_SECONDS` and `CONCURRENCY_LIMIT`: website availability check controls.

In [80]:
from urllib.parse import urlparse, urlunparse
from pathlib import Path
import asyncio
import re
import unicodedata

import httpx
import pandas as pd
import tldextract

PROJECT_DIR = Path.cwd().resolve()

# Works when running from a notebooks/ folder in a normal repo.
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_COMPANY_PATH = PROJECT_DIR / "data" / "raw" / "dataset_country_test.csv"

# Works when running in Colab with the CSV uploaded beside the notebook.
if not RAW_COMPANY_PATH.exists():
    RAW_COMPANY_PATH = Path("dataset_country_test.csv")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CLEANED_TARGET_COUNTRIES_PATH = PROCESSED_DIR / "company_cleaned_target_countries.csv"
OUTPUT_DOMAIN_CHECKED_PATH = PROCESSED_DIR / "company_domain_checked.csv"
OUTPUT_DUPLICATE_DOMAIN_REVIEW_PATH = PROCESSED_DIR / "company_duplicate_domain_review.csv"
OUTPUT_REACHABLE_PATH = PROCESSED_DIR / "company_reachable_websites.csv"
OUTPUT_UNREACHABLE_PATH = PROCESSED_DIR / "company_unreachable_websites_reserve.csv"
OUTPUT_GOVERNMENT_SUMMARY_PATH = PROCESSED_DIR / "government_ratio_summary.csv"
OUTPUT_SEED_SAMPLE_PATH = PROCESSED_DIR / "company_seed_sample.csv"
OUTPUT_SCRAPE_READY_PATH = PROCESSED_DIR / "company_scrape_ready.csv"


TARGET_COUNTRIES = [
    "australia",
    "germany",
    "south africa",
    "brazil",
    "turkey",
    "france",
    "saudi arabia",
    "china",
    "japan",
    "sri lanka",
]

COUNTRY_DOMAIN_SUFFIXES = {
    "australia": "au",
    "germany": "de",
    "south africa": "za",
    "brazil": "br",
    "turkey": "tr",
    "france": "fr",
    "saudi arabia": "sa",
    "china": "cn",
    "japan": "jp",
    "sri lanka": "lk",
}

LEGAL_COMPANY_WORDS = {
    "pty",
    "ltd",
    "limited",
    "corp",
    "corporation",
    "inc",
    "company",
    "co",
    "group",
    "holdings",
    "plc",
    "llc",
    "gmbh",
    "ag",
    "sa",
    "sarl",
    "bv",
    "nv",
}

BAD_DOMAIN_PATTERNS = [
    "instagram.com",
    "facebook.com",
    "facebook.com.br",
    "fb.me",
    "linkedin.com",
    "twitter.com",
    "x.com",
    "youtube.com",
    "youtu.be",
    "tiktok.com",
    "linktr.ee",
    "lnk.bio",
    "campsite.bio",
    "instabio.cc",
    "bio.link",
    "wa.me",
    "weibo.com",
    "qq.com",
    "weebly.com",
    "wixsite.com",
    "wordpress.com",
    "blogspot.com",
    "business.site",
    "negocio.site",
    "sites.google.com",
    "g.page",
    "business.google.com",
    "medium.com",
    "github.com",
    "github.io",
    "behance.net",
    "vimeo.com",
    "soundcloud.com",
    "spotify.com",
    "anchor.fm",
    "ausha.co",
    "meetup.com",
    "calendly.com",
    "forms.gle",
    "docs.google.com",
    "drive.google.com",
    "mailchi.mp",
    "etsy.com",
    "mercadolibre",
    "1688.com",
    "wikipedia.org",
]

BAD_FILE_EXTENSIONS = [
    ".pdf",
    ".jpg",
    ".jpeg",
    ".png",
    ".gif",
    ".svg",
    ".webp",
    ".doc",
    ".docx",
    ".xls",
    ".xlsx",
    ".ppt",
    ".pptx",
    ".zip",
    ".rar",
]

# The dataset does not provide exact headcount.
# 51-200 is retained as an approximate match to the 100+ staff requirement.
SIZE_CATEGORY_MAP = {
    "51-200": "small",
    "201-500": "small",
    "501-1000": "medium",
    "1001-5000": "large",
    "5001-10000": "enterprise",
    "10001+": "enterprise",
}

MAX_FINAL_ROWS = 5000

MAX_PER_COUNTRY = MAX_FINAL_ROWS // len(TARGET_COUNTRIES)

MAX_COMPANIES_PER_COUNTRY = MAX_PER_COUNTRY

SIZE_TARGETS = {
    "small": MAX_PER_COUNTRY // 4,
    "medium": MAX_PER_COUNTRY // 4,
    "large": MAX_PER_COUNTRY // 4,
    "enterprise": MAX_PER_COUNTRY // 4,
}

# Allows diversity without blocking the sample at around 500 rows.
MAX_COMPANIES_PER_SECTOR_PER_SIZE = 15
MAX_PER_SECTOR_PER_COUNTRY = 25

GOVERNMENT_SECTOR = "Government, Public Sector & Civic"
MAX_GOVERNMENT_RATIO = 0.10
MAX_GOVERNMENT_PER_COUNTRY = int(MAX_PER_COUNTRY * MAX_GOVERNMENT_RATIO)

REQUEST_TIMEOUT_SECONDS = 10
CONCURRENCY_LIMIT = 20

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("Raw input path:", RAW_COMPANY_PATH)
print("Processed output directory:", PROCESSED_DIR)
print("Target final rows:", MAX_FINAL_ROWS)
print("Target per country:", MAX_PER_COUNTRY)
print("Max government per country:", MAX_GOVERNMENT_PER_COUNTRY)

Raw input path: /Users/namanhtran/project/testing_ground/data/raw/dataset_country_test.csv
Processed output directory: /Users/namanhtran/project/testing_ground/data/processed
Target final rows: 5000
Target per country: 500
Max government per country: 50


## 3. Basic cleaning helper functions

These functions clean column names, text values, website URLs, domains, and size buckets.

In [81]:
def clean_col_names(dataframe):
    """Standardise DataFrame column names."""
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return dataframe


def clean_text_columns(dataframe, columns):
    """Trim and lowercase selected text columns if they exist."""
    for column in columns:
        if column in dataframe.columns:
            dataframe[column] = (
                dataframe[column]
                .astype("string")
                .str.strip()
                .str.lower()
            )
    return dataframe


def normalise_website_url(website):
    """Convert a raw website value into a URL with a protocol."""
    if pd.isna(website):
        return None

    website = str(website).strip()

    if not website:
        return None

    if website.startswith("http://") or website.startswith("https://"):
        return website

    return f"https://{website}"


def extract_domain(url):
    """Extract a lowercase domain and remove the leading www prefix."""
    url = normalise_website_url(url)

    if url is None:
        return None

    parsed_url = urlparse(url)
    domain = parsed_url.netloc.lower().strip()

    if domain.startswith("www."):
        domain = domain[4:]

    return domain if domain else None


def is_website_format_usable(website):
    """Return True when a website value looks like a usable company website."""
    if pd.isna(website):
        return False

    website = str(website).strip().lower()

    if not website:
        return False

    if "@" in website:
        return False

    if "." not in website:
        return False

    for extension in BAD_FILE_EXTENSIONS:
        if website.endswith(extension):
            return False

    domain = extract_domain(website)

    if domain is None:
        return False

    for pattern in BAD_DOMAIN_PATTERNS:
        if domain == pattern or domain.endswith("." + pattern):
            return False

    return True


def map_company_size(size):
    """Map a raw employee-size value into a simple size bucket."""
    size = str(size).strip().lower()
    return SIZE_CATEGORY_MAP.get(size, "too_small")


def build_url_variants(url):
    """Build URL variants with and without www."""
    url = normalise_website_url(url)

    if url is None:
        return []

    parsed_url = urlparse(url)
    scheme = parsed_url.scheme or "https"
    domain = parsed_url.netloc.lower().strip()
    path = parsed_url.path or ""

    if not domain:
        return []

    variants = []
    variants.append(urlunparse((scheme, domain, path, "", "", "")))

    if domain.startswith("www."):
        non_www_domain = domain.replace("www.", "", 1)
        variants.append(urlunparse((scheme, non_www_domain, path, "", "", "")))
    else:
        www_domain = f"www.{domain}"
        variants.append(urlunparse((scheme, www_domain, path, "", "", "")))

    return list(dict.fromkeys(variants))

## 4. Country-domain signal helper

This is only a first-pass domain suffix signal.

It does not prove headquarters country. It only checks whether the domain suffix appears consistent with the dataset country.

In [82]:
def classify_country_website(country, domain):
    """
    Classify whether a domain suffix appears to match the dataset country.

    country_match: country-specific suffix matches the dataset country.
    country_mismatch: suffix appears country-specific but points to another country.
    global_or_unknown: generic suffix such as .com, .org, or no reliable country signal.
    """
    if pd.isna(country) or pd.isna(domain):
        return "global_or_unknown"

    country_key = str(country).strip().lower()
    extracted = tldextract.extract(str(domain).lower().strip())
    suffix = extracted.suffix

    if not suffix:
        return "global_or_unknown"

    expected_tld = COUNTRY_DOMAIN_SUFFIXES.get(country_key)
    last_suffix_part = suffix.rsplit(".", 1)[-1]

    if expected_tld is None:
        return "global_or_unknown"

    if last_suffix_part == expected_tld:
        return "country_match"

    if len(last_suffix_part) == 2:
        return "country_mismatch"

    return "global_or_unknown"

## 5. Industry mapping helper

This maps raw industries into broader sectors so the final sample is not dominated by one industry type.

In [83]:
sector_to_industries = {
    "Agriculture, Food & Beverage": "dairy; farming; food & beverages; food production; wine and spirits",
    "Arts, Media & Entertainment": "writing and editing; animation; broadcast media; design; entertainment; fine art; media production; motion pictures and film; newspapers; online media; photography; printing; publishing; performing arts; music; computer games; museums and institutions; graphic design",
    "Automotive, Aerospace & Transportation Equipment": "automotive; aviation & aerospace; defense & space; shipbuilding",
    "Business Services & Administration": "capital markets; business supplies and equipment; executive office; facilities services; management consulting; outsourcing/offshoring; program development; staffing and recruiting",
    "Construction, Real Estate & Built Environment": "architecture & planning; building materials; civil engineering; commercial real estate; construction; real estate",
    "Consumer Goods & Retail": "apparel & fashion; consumer electronics; consumer goods; consumer services; cosmetics; furniture; luxury goods & jewelry; retail; sporting goods; supermarkets; textiles; wholesale",
    "Education & Research": "e-learning; education management; higher education; libraries; primary/secondary education; professional training & coaching; research",
    "Energy, Environment & Utilities": "environmental services; oil & energy; renewables & environment; utilities",
    "Financial Services & Insurance": "accounting; banking; financial services; insurance; investment banking; investment management; venture capital & private equity",
    "Government, Public Sector & Civic": "public policy; railroad manufacture; recreational facilities and service; government administration; government relations; international affairs; legislative office; military; political organization; think tanks; public relations and communications",
    "Healthcare & Life Sciences": "veterinary; alternative medicine; biotechnology; health, wellness and fitness; hospital & health care; medical devices; medical practice; mental health care; pharmaceuticals",
    "Hospitality, Travel & Leisure": "airlines/aviation; gambling & casinos; hospitality; leisure, travel & tourism; restaurants; sports",
    "Industrial Manufacturing & Materials": "chemicals; electrical/electronic manufacturing; glass, ceramics & concrete; industrial automation; machinery; mechanical or industrial engineering; mining & metals; packaging and containers; paper & forest products; plastics; semiconductors",
    "Information Technology & Internet": "computer & network security; computer hardware; computer networking; computer software; information services; information technology and services; internet; telecommunications; wireless; nanotechnology",
    "Legal & Justice": "judiciary; law enforcement; law practice; legal services; public safety; security and investigations",
    "Logistics, Transportation & Trade": "import and export; international trade and development; logistics and supply chain; maritime; package/freight delivery; transportation/trucking/railroad; warehousing",
    "Marketing, Communications & Events": "market research; events services; marketing and advertising; translation and localization",
    "Nonprofit, Social Services & Religion": "fund-raising; civic & social organization; individual & family services; non-profit organization management; religious institutions",
}

industry_to_sector = {
    industry.strip().lower(): sector
    for sector, industries in sector_to_industries.items()
    for industry in industries.split(";")
}

## 6. Sampling helper function

This creates an initial candidate pool before website availability checking.

It samples within each country while balancing size buckets and reducing sector dominance.

In [84]:
def sample_country_by_size_and_sector(group):
    """Sample one country while balancing size buckets and limiting sector dominance."""
    sampled_groups = []

    for size_bucket, target_count in SIZE_TARGETS.items():
        size_group = group[group["size_bucket"] == size_bucket].copy()

        if size_group.empty:
            continue

        sector_samples = []

        for _, sector_group in size_group.groupby("High_Level_Sector"):
            sampled_sector = sector_group.sample(
                n=min(MAX_COMPANIES_PER_SECTOR_PER_SIZE, len(sector_group)),
                random_state=42,
            )
            sector_samples.append(sampled_sector)

        if not sector_samples:
            continue

        sampled = pd.concat(sector_samples)

        if len(sampled) > target_count:
            sampled = sampled.sample(n=target_count, random_state=42)

        if len(sampled) < target_count:
            remaining_needed = target_count - len(sampled)
            remaining_pool = size_group.drop(sampled.index, errors="ignore")

            if not remaining_pool.empty:
                top_up = remaining_pool.sample(
                    n=min(remaining_needed, len(remaining_pool)),
                    random_state=42,
                )
                sampled = pd.concat([sampled, top_up])

        sampled_groups.append(sampled)

    if not sampled_groups:
        return group.sample(
            n=min(MAX_COMPANIES_PER_COUNTRY, len(group)),
            random_state=42,
        )

    sampled_country = pd.concat(sampled_groups)

    if len(sampled_country) > MAX_COMPANIES_PER_COUNTRY:
        sampled_country = sampled_country.sample(
            n=MAX_COMPANIES_PER_COUNTRY,
            random_state=42,
        )

    return sampled_country

## 7. Domain-name matching helper functions

This rule-based layer checks whether the company name and website domain appear to match.

The status rule is:

| Score | Status |
|---:|---|
| 80 or above | keep |
| 50 to 79 | review |
| Below 50 | remove |

Only `keep` records move forward automatically.

In [85]:
def normalise_to_ascii(text):
    """Convert accented Latin characters to ASCII where possible."""
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return text


def norm_company_token(company_name):
    """Clean a company name into useful comparison tokens."""
    company_name = normalise_to_ascii(company_name)
    company_name = re.sub(r"[^a-z0-9]+", " ", company_name)

    tokens = company_name.split()

    company_tokens = [
        token
        for token in tokens
        if token not in LEGAL_COMPANY_WORDS and len(token) > 1
    ]
    return company_tokens


def norm_domain_token(domain_name):
    """Clean a website domain into useful comparison tokens."""
    domain_name = str(domain_name).lower().strip()
    extracted = tldextract.extract(domain_name)
    registered_domain = normalise_to_ascii(extracted.domain)

    domain_tokens = re.split(r"[\-. _]+", registered_domain)
    domain_tokens = [
        token
        for token in domain_tokens
        if token and len(token) > 1
    ]
    return domain_tokens


def compute_match_score(company_tokens, domain_tokens):
    """Return 100 when at least one company token exactly matches a domain token."""
    matched_tokens = set()

    for company_token in company_tokens:
        for domain_token in domain_tokens:
            if company_token == domain_token:
                matched_tokens.add(company_token)

    if matched_tokens:
        return 100, matched_tokens

    return 0, matched_tokens


def compute_substring_matching(company_tokens, domain_tokens, already_matched):
    """Score company tokens that appear inside domain tokens, or the reverse."""
    matched_tokens = set()

    for company_token in company_tokens:
        if company_token in already_matched:
            continue

        if len(company_token) < 4:
            continue

        for domain_token in domain_tokens:
            if company_token in domain_token or domain_token in company_token:
                matched_tokens.add(company_token)
                break

    if not matched_tokens:
        return 0, matched_tokens

    if len(matched_tokens) == len(company_tokens):
        return 100, matched_tokens

    return 80, matched_tokens


def compute_abbreviation_score(domain_tokens, company_tokens):
    """Score domains that match or start with the company-name abbreviation."""
    if len(company_tokens) < 2:
        return 0

    abbreviation = "".join(token[0] for token in company_tokens)

    for domain_token in domain_tokens:
        if domain_token == abbreviation:
            return 90

        if domain_token.startswith(abbreviation):
            return 70

    return 0


def classify_domain_match_score(final_score):
    """Convert a domain match score into keep, review, or remove."""
    if final_score >= 80:
        return "keep"

    if final_score >= 50:
        return "review"

    return "remove"


def compute_final(company_name, website_url):
    """Compute the domain match scores and final domain status for one row."""
    company_tokens = norm_company_token(company_name)
    domain_tokens = norm_domain_token(website_url)

    if not company_tokens or not domain_tokens:
        return {
            "company_tokens": company_tokens,
            "domain_tokens": domain_tokens,
            "full_match_score": 0,
            "substring_match_score": 0,
            "abbreviation_score": 0,
            "final_score": 0,
            "domain_match_status": "remove",
        }

    exact_score, exact_tokens = compute_match_score(company_tokens, domain_tokens)

    if exact_score >= 100:
        final_score = exact_score
        return {
            "company_tokens": company_tokens,
            "domain_tokens": domain_tokens,
            "full_match_score": exact_score,
            "substring_match_score": 0,
            "abbreviation_score": 0,
            "final_score": final_score,
            "domain_match_status": classify_domain_match_score(final_score),
        }

    substring_score, substring_tokens = compute_substring_matching(
        company_tokens,
        domain_tokens,
        exact_tokens,
    )

    if substring_score >= 80:
        final_score = substring_score
        return {
            "company_tokens": company_tokens,
            "domain_tokens": domain_tokens,
            "full_match_score": 0,
            "substring_match_score": substring_score,
            "abbreviation_score": 0,
            "final_score": final_score,
            "domain_match_status": classify_domain_match_score(final_score),
        }

    abbreviation_score = compute_abbreviation_score(domain_tokens, company_tokens)
    final_score = abbreviation_score

    return {
        "company_tokens": company_tokens,
        "domain_tokens": domain_tokens,
        "full_match_score": 0,
        "substring_match_score": 0,
        "abbreviation_score": abbreviation_score,
        "final_score": final_score,
        "domain_match_status": classify_domain_match_score(final_score),
    }

## 8. Website availability helper functions

The availability check confirms that the website returns reachable HTML.

A website passes only when:

1. The request succeeds.
2. The HTTP status code is below 400.
3. The response content type contains `text/html`.

In [86]:
async def check_website(client, url, semaphore):
    """Check one website and return a structured availability result."""
    url_variants = build_url_variants(url)
    last_error = None

    async with semaphore:
        for candidate_url in url_variants:
            try:
                response = await client.get(
                    candidate_url,
                    headers=HEADERS,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                    follow_redirects=True,
                )

                status_code = response.status_code
                content_type = response.headers.get("Content-Type", "").lower()

                if status_code >= 400:
                    last_error = f"http_error: {status_code}"
                    continue

                if "text/html" not in content_type:
                    last_error = f"not_html_error: {content_type}"
                    continue

                return {
                    "is_website_reachable": True,
                    "final_url": str(response.url),
                    "status_code": status_code,
                    "content_type": content_type,
                    "failure_reason": None,
                    "checked_url_variants": url_variants,
                }

            except httpx.TimeoutException:
                last_error = "timeout"
            except httpx.ConnectError:
                last_error = "connection_error"
            except httpx.TooManyRedirects:
                last_error = "too_many_redirects"
            except Exception as error:
                last_error = f"unexpected_error: {str(error)[:100]}"

    return {
        "is_website_reachable": False,
        "final_url": None,
        "status_code": None,
        "content_type": None,
        "failure_reason": last_error,
        "checked_url_variants": url_variants,
    }


async def run_avail_check(dataframe):
    """Run website availability checks for every website_url in a DataFrame."""
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    async with httpx.AsyncClient() as client:
        tasks = [
            check_website(
                client=client,
                url=website_url,
                semaphore=semaphore,
            )
            for website_url in dataframe["website_url"]
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)

## 9. Final sampling and validation helpers

This final sampling step explicitly enforces the government limit.

Each country should contribute up to 50 final companies.

Government organisations are capped at 5 per country, which equals 10% of 50.

In [87]:
def build_final_scrape_ready_sample(df_qualified):
    """
    Build the final scrape-ready dataset.

    Enforces:
    - 50 companies per country where enough reachable companies exist.
    - Maximum 5 government organisations per country.
    - Maximum 5 companies from any single sector per country.
    """
    sampled_groups = []

    for _, country_group in df_qualified.groupby("country"):
        country_sampled = []
        sector_counts = {}
        government_count = 0

        size_sector_queues = [
            sub_group.sample(frac=1, random_state=42)
            for _, sub_group in country_group.groupby(["size_bucket", "High_Level_Sector"])
        ]

        while len(country_sampled) < MAX_PER_COUNTRY and size_sector_queues:
            active_queues = []

            for queue in size_sector_queues:
                if len(country_sampled) >= MAX_PER_COUNTRY:
                    break

                if queue.empty:
                    continue

                row = queue.iloc[0]
                sector = row["High_Level_Sector"]

                current_sector_count = sector_counts.get(sector, 0)
                is_government = sector == GOVERNMENT_SECTOR

                can_add_sector = current_sector_count < MAX_PER_SECTOR_PER_COUNTRY
                can_add_government = (
                    not is_government
                    or government_count < MAX_GOVERNMENT_PER_COUNTRY
                )

                if can_add_sector and can_add_government:
                    country_sampled.append(row)
                    sector_counts[sector] = current_sector_count + 1

                    if is_government:
                        government_count += 1

                remaining_queue = queue.iloc[1:]

                if not remaining_queue.empty:
                    active_queues.append(remaining_queue)

            size_sector_queues = active_queues

        if country_sampled:
            sampled_groups.append(pd.DataFrame(country_sampled))

    if sampled_groups:
        return pd.concat(sampled_groups, ignore_index=True)

    return pd.DataFrame(columns=df_qualified.columns)


def build_government_summary(dataframe):
    """Build country-level and overall government ratio summaries."""
    if dataframe.empty:
        return pd.DataFrame(
            columns=["country", "total_companies", "government_companies", "government_ratio"]
        )

    government_summary = (
        dataframe
        .assign(is_government=lambda df: df["High_Level_Sector"].eq(GOVERNMENT_SECTOR))
        .groupby("country")
        .agg(
            total_companies=("name", "count"),
            government_companies=("is_government", "sum"),
        )
        .reset_index()
    )

    government_summary["government_ratio"] = (
        government_summary["government_companies"]
        / government_summary["total_companies"]
    )

    return government_summary


def validate_government_ratio(dataframe, government_summary):
    """Assert that government ratio is within the project requirement."""
    if dataframe.empty:
        raise ValueError("Final dataset is empty. Cannot validate government ratio.")

    if not government_summary.empty:
        max_country_ratio = government_summary["government_ratio"].max()
        assert (
            max_country_ratio <= MAX_GOVERNMENT_RATIO + 1e-12
        ), "Government ratio exceeds 10% in at least one country."

    total_government_count = dataframe["High_Level_Sector"].eq(GOVERNMENT_SECTOR).sum()
    total_government_ratio = total_government_count / len(dataframe)

    assert (
        total_government_ratio <= MAX_GOVERNMENT_RATIO + 1e-12
    ), "Overall government ratio exceeds 10%."

    print("Total government count:", total_government_count)
    print("Total government ratio:", total_government_ratio)

## 10. Load raw company data

Input file expected in the normal repo structure:

`../data/raw/dataset_country_test.csv`

If running in Colab, upload `dataset_country_test.csv` beside the notebook.

In [88]:
comp_df = pd.read_csv(RAW_COMPANY_PATH, header=0)
comp_df = clean_col_names(comp_df)
display(comp_df.head())

,name,website,country,industry,founded,size
0,onehealth,onehealthconsultinggroup.org,united states,NaN,2020.0,51-200
1,grupo holco,grupoholco.com,ecuador,logistics and supply chain,1939.0,51-200
2,list fashion group s.p.a.,listfashiongroup.it,italy,apparel & fashion,NaN,51-200
3,city-kej produkter ab,city-kej.se,sweden,retail,1952.0,11-50
4,deltadots,deltadots.com,united states,NaN,2021.0,11-50


## 11. Remove rows with missing required fields

A company is kept only if it has:

- `name`
- `country`
- `size`
- `website`
- `industry`

In [89]:
df_cleaned = comp_df[
    comp_df["name"].notna()
    & comp_df["country"].notna()
    & comp_df["size"].notna()
    & comp_df["website"].notna()
    & comp_df["industry"].notna()
].copy()

df_cleaned = clean_text_columns(
    df_cleaned,
    ["name", "website", "country", "industry", "size"],
)

print("Rows after required-field filter:", len(df_cleaned))
display(df_cleaned.head())

Rows after required-field filter: 11978009


,name,website,country,industry,founded,size
1,grupo holco,grupoholco.com,ecuador,logistics and supply chain,1939.0,51-200
2,list fashion group s.p.a.,listfashiongroup.it,italy,apparel & fashion,NaN,51-200
3,city-kej produkter ab,city-kej.se,sweden,retail,1952.0,11-50
5,simplifyit group,simplifyitgroup.com,united states,information technology and services,NaN,1-10
6,forest of hope,pikminwiki.com,united states,non-profit organization management,NaN,1-10


## 12. Keep only target countries

This keeps only the countries listed in `TARGET_COUNTRIES`.

In [90]:
df_target_countries = df_cleaned[
    df_cleaned["country"].isin(TARGET_COUNTRIES)
].copy()

country_counts = (
    df_target_countries["country"]
    .value_counts()
    .rename_axis("country")
    .reset_index(name="row_count")
)

print("Rows after target-country filter:", len(df_target_countries))
display(country_counts)

Rows after target-country filter: 2723784


,country,row_count
0,brazil,818684
1,france,492900
2,germany,484363
3,australia,459320
4,china,168981
5,turkey,138868
6,south africa,92768
7,japan,30071
8,saudi arabia,25525
9,sri lanka,12304


## 13. Filter unusable website formats

This removes obvious non-company websites before doing network requests.

In [91]:
df_target_countries["website_format_usable"] = (
    df_target_countries["website"].apply(is_website_format_usable)
)

df_website_usable = df_target_countries[
    df_target_countries["website_format_usable"] == True
].copy()

df_website_usable["website_url"] = (
    df_website_usable["website"].apply(normalise_website_url)
)
df_website_usable["domain"] = (
    df_website_usable["website_url"].apply(extract_domain)
)

df_website_usable = df_website_usable[
    df_website_usable["domain"].notna()
].copy()

print("Website format usable counts:")
display(df_target_countries["website_format_usable"].value_counts(dropna=False))

print("Rows after website-format filter:", len(df_website_usable))
display(df_website_usable.head())

Website format usable counts:


website_format_usable
True     2597840
False     125944
Name: count, dtype: int64

Rows after website-format filter: 2597840


,name,website,country,industry,founded,size,website_format_usable,website_url,domain
12,izeau,izeau.fr,france,environmental services,2019.0,1-10,True,https://izeau.fr,izeau.fr
21,rei dos alevinos,reidosalevinos.com.br,brazil,retail,NaN,1-10,True,https://reidosalevinos.com.br,reidosalevinos.com.br
23,au fil de l'or,aufildelor.fr,france,retail,NaN,1-10,True,https://aufildelor.fr,aufildelor.fr
27,haktrak,haktrak.com,saudi arabia,computer & network security,2019.0,11-50,True,https://haktrak.com,haktrak.com
30,ouroboros design & furniture,ouroborosdesign.fr,france,retail,NaN,1-10,True,https://ouroborosdesign.fr,ouroborosdesign.fr


## 14. Map company size into buckets

Rows mapped to `too_small` are removed.

Important: `51-200` is kept as `small` because the dataset does not provide exact headcount.

In [92]:
df_website_usable["size_bucket"] = (
    df_website_usable["size"].apply(map_company_size)
)

df_size_valid = df_website_usable[
    df_website_usable["size_bucket"] != "too_small"
].copy()

size_summary = (
    df_size_valid
    .groupby(["country", "size_bucket"])
    .size()
    .reset_index(name="company_count")
    .sort_values(["country", "size_bucket"])
)

print("Rows after size filter:", len(df_size_valid))
display(size_summary)

Rows after size filter: 324940


,country,size_bucket,company_count
0,australia,enterprise,754
1,australia,large,1796
2,australia,medium,2197
3,australia,small,28135
4,brazil,enterprise,1705
5,brazil,large,4775
6,brazil,medium,5414
7,brazil,small,63085
8,china,enterprise,1850
9,china,large,3873


## 15. Map industry into high-level sectors

Unmapped industries are assigned to `Other`.

In [93]:
df_size_valid["industry_clean"] = (
    df_size_valid["industry"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mapped_sector = df_size_valid["industry_clean"].map(industry_to_sector)

unmapped_industries = (
    df_size_valid.loc[mapped_sector.isna(), "industry"]
    .dropna()
    .sort_values()
    .unique()
)

df_size_valid["High_Level_Sector"] = mapped_sector.fillna("Other")

df_industry_fixed = df_size_valid.drop(columns=["industry_clean"])

print("Unmapped industry count:", len(unmapped_industries))
if len(unmapped_industries) > 0:
    print("Sample unmapped industries:")
    print(unmapped_industries[:50])

sector_summary = (
    df_industry_fixed["High_Level_Sector"]
    .value_counts(dropna=False)
    .reset_index()
)
sector_summary.columns = ["High_Level_Sector", "company_count"]

display(sector_summary)
display(df_industry_fixed.head())

Unmapped industry count: 8
Sample unmapped industries:
<StringArray>
[      'alternative dispute resolution',
                      'arts and crafts',
                              'fishery',
                      'human resources',
                         'philanthropy',
                             'ranching',
 'recreational facilities and services',
                              'tobacco']
Length: 8, dtype: string


,High_Level_Sector,company_count
0,Industrial Manufacturing & Materials,46876
1,Consumer Goods & Retail,40488
2,"Construction, Real Estate & Built Environment",35828
3,Information Technology & Internet,30428
4,Healthcare & Life Sciences,17537
5,"Logistics, Transportation & Trade",16814
6,Education & Research,16647
7,Financial Services & Insurance,15025
8,Business Services & Administration,14804
9,"Hospitality, Travel & Leisure",12672


,name,website,country,industry,founded,size,website_format_usable,website_url,domain,size_bucket,High_Level_Sector
142,total links,totallinks.com.br,brazil,telecommunications,1993.0,51-200,True,https://totallinks.com.br,totallinks.com.br,small,Information Technology & Internet
526,cologne institute for information systems (ciis),ciis.uni-koeln.de,germany,research,2018.0,51-200,True,https://ciis.uni-koeln.de,ciis.uni-koeln.de,small,Education & Research
580,cipalam indústria e comércio de laminados ltda.,cipalam.com.br,brazil,mechanical or industrial engineering,1986.0,201-500,True,https://cipalam.com.br,cipalam.com.br,small,Industrial Manufacturing & Materials
600,secretaria de saúde do distrito federal,saude.df.gov.br,brazil,utilities,1957.0,10001+,True,https://saude.df.gov.br,saude.df.gov.br,enterprise,"Energy, Environment & Utilities"
726,ebs.invest e.v.,ebsinvest.com,germany,investment management,1981.0,51-200,True,https://ebsinvest.com,ebsinvest.com,small,Financial Services & Insurance


## 16. Build pre-check dataset and apply domain matching

This keeps the fields needed for domain matching and website checking.

Only `keep` records move forward automatically.

In [94]:
final_columns = [
    "name",
    "country",
    "industry",
    "High_Level_Sector",
    "size",
    "size_bucket",
    "website_url",
    "domain",
]

df_final = df_industry_fixed[final_columns].copy()

df_final["company_count_for_country_in_pool"] = (
    df_final
    .groupby("country")["name"]
    .transform("count")
)

df_final["total_countries_in_pool"] = df_final["country"].nunique()

domain_match_results = df_final.apply(
    lambda row: compute_final(
        company_name=row["name"],
        website_url=row["website_url"],
    ),
    axis=1,
)

domain_match_df = pd.DataFrame(domain_match_results.tolist())

df_domain_checked = pd.concat(
    [
        df_final.reset_index(drop=True),
        domain_match_df.reset_index(drop=True),
    ],
    axis=1,
)

df_domain_keep = df_domain_checked[
    df_domain_checked["domain_match_status"] == "keep"
].copy()

df_domain_review = df_domain_checked[
    df_domain_checked["domain_match_status"] == "review"
].copy()

df_domain_remove = df_domain_checked[
    df_domain_checked["domain_match_status"] == "remove"
].copy()

print("Domain match status counts:")
display(df_domain_checked["domain_match_status"].value_counts(dropna=False))

print("Rows kept after domain matching:", len(df_domain_keep))

Domain match status counts:


domain_match_status
keep      264601
remove     58299
review      2040
Name: count, dtype: int64

Rows kept after domain matching: 264601


## 17. Review duplicate/shared domains and apply country-domain signal

This block does two things:

1. Removes exact duplicate company/domain records.
2. Separates suspicious shared domains and country-domain mismatches into a review file.

Only safer automatic candidates are used for sampling.

In [95]:
df_domain_candidates = (
    df_domain_keep
    .drop_duplicates(
        subset=["name", "country", "domain"],
        keep="first",
    )
    .copy()
)

df_domain_candidates["country_website_status"] = (
    df_domain_candidates.apply(
        lambda row: classify_country_website(
            country=row["country"],
            domain=row["domain"],
        ),
        axis=1,
    )
)

website_priority = {
    "country_match": 2,
    "global_or_unknown": 1,
    "country_mismatch": 0,
}

df_domain_candidates["country_website_priority"] = (
    df_domain_candidates["country_website_status"].map(website_priority)
)

# Prefer a country-specific domain over a generic domain when the same company appears more than once.
df_domain_candidates = (
    df_domain_candidates
    .sort_values(
        ["name", "country", "country_website_priority"],
        ascending=[True, True, False],
    )
    .drop_duplicates(
        subset=["name", "country"],
        keep="first",
    )
    .copy()
)

df_domain_candidates["companies_using_domain"] = (
    df_domain_candidates
    .groupby("domain")["name"]
    .transform("nunique")
)

df_domain_candidates["countries_using_domain"] = (
    df_domain_candidates
    .groupby("domain")["country"]
    .transform("nunique")
)

df_duplicate_domain_review = df_domain_candidates[
    (df_domain_candidates["companies_using_domain"] > 1)
    | (df_domain_candidates["countries_using_domain"] > 1)
    | (df_domain_candidates["country_website_status"] == "country_mismatch")
].copy()

df_auto_candidates = df_domain_candidates[
    (df_domain_candidates["companies_using_domain"] == 1)
    & (df_domain_candidates["countries_using_domain"] == 1)
    & (df_domain_candidates["country_website_status"] != "country_mismatch")
].copy()

print("Automatic candidates:", len(df_auto_candidates))
print("Records requiring duplicate/domain review:", len(df_duplicate_domain_review))

display(
    df_duplicate_domain_review[
        [
            "name",
            "country",
            "website_url",
            "domain",
            "country_website_status",
            "companies_using_domain",
            "countries_using_domain",
        ]
    ].head(50)
)

Automatic candidates: 258149
Records requiring duplicate/domain review: 5937


,name,country,website_url,domain,country_website_status,companies_using_domain,countries_using_domain
234730,#burgerlove,australia,https://burgerlove.co,burgerlove.co,country_mismatch,1,1
98986,1000s jewelry,china,https://jewelry.lc,jewelry.lc,country_mismatch,1,1
269806,12min.me ⏰,germany,https://12min.me,12min.me,country_mismatch,1,1
52715,180.ai,china,https://180.ai,180.ai,country_mismatch,1,1
205653,20 nations league of blockchain(b20),china,https://b20.io,b20.io,country_mismatch,1,1
18061,21diamonds,germany,https://21diamonds.es,21diamonds.es,country_mismatch,1,1
306086,23grad - netzwerk umwelt- und nachhaltigkeitsw...,germany,https://23grad.eu,23grad.eu,country_mismatch,1,1
235257,24 25 tv & medienproduktion gmbh,germany,https://24-25.tv,24-25.tv,country_mismatch,1,1
93083,24timer,australia,https://24.dk,24.dk,country_mismatch,1,1
241705,2m language services,france,https://2m.com.au,2m.com.au,country_mismatch,1,1


## 18. Initial balanced candidate sample before website checks

This creates a 100-company candidate pool per country before availability checking.

The larger pool gives the final step enough reachable websites to choose 50 per country.

In [96]:
sampled_country_frames = []

for _, country_group in df_auto_candidates.groupby("country"):
    sampled_country_frames.append(sample_country_by_size_and_sector(country_group))

if sampled_country_frames:
    df_sampled = pd.concat(sampled_country_frames, ignore_index=True)
else:
    df_sampled = pd.DataFrame(columns=df_auto_candidates.columns)

print("Initial candidate rows:", len(df_sampled))

country_summary = (
    df_sampled
    .groupby("country")
    .size()
    .reset_index(name="company_count")
)

display(country_summary)

size_summary = (
    df_sampled
    .groupby(["country", "size_bucket"])
    .size()
    .reset_index(name="company_count")
    .sort_values(["country", "size_bucket"])
)

display(size_summary)

Initial candidate rows: 4964


,country,company_count
0,australia,500
1,brazil,500
2,china,500
3,france,500
4,germany,500
5,japan,500
6,saudi arabia,500
7,south africa,500
8,sri lanka,464
9,turkey,500


,country,size_bucket,company_count
0,australia,enterprise,125
1,australia,large,125
2,australia,medium,125
3,australia,small,125
4,brazil,enterprise,125
5,brazil,large,125
6,brazil,medium,125
7,brazil,small,125
8,china,enterprise,125
9,china,large,125


## 19. Run website availability checks

This checks the sampled candidate websites asynchronously.

In [ ]:
availability_df = await run_avail_check(df_sampled)

availability_columns = [
    "is_website_reachable",
    "final_url",
    "status_code",
    "content_type",
    "failure_reason",
    "checked_url_variants",
]

df_sampled = df_sampled.drop(
    columns=availability_columns,
    errors="ignore",
)

df_checked = pd.concat(
    [
        df_sampled.reset_index(drop=True),
        availability_df.reset_index(drop=True),
    ],
    axis=1,
)

df_checked["scraping_ready"] = (
    (df_checked["is_website_reachable"] == True)
    & (df_checked["final_url"].notna())
    & (df_checked["content_type"].str.contains("text/html", na=False))
    & (df_checked["domain_match_status"] == "keep")
)

df_reachable = df_checked[df_checked["is_website_reachable"] == True].copy()
df_unreachable = df_checked[df_checked["is_website_reachable"] == False].copy()
df_qualified = df_checked[df_checked["scraping_ready"] == True].copy()

print("Checked rows:", len(df_checked))
print("Reachable rows:", len(df_reachable))
print("Qualified scrape-ready candidates:", len(df_qualified))

## 20. Build final scrape-ready dataset with explicit government limit

The final sample enforces:

- Up to 50 companies per country.
- Maximum 5 government organisations per country.
- Maximum 5 companies from any one sector per country.

This is the main fix for the government `< 10%` requirement.

In [ ]:
df_scrape_ready = build_final_scrape_ready_sample(df_qualified)

df_scrape_ready = (
    df_scrape_ready
    .sort_values(["country", "size_bucket", "High_Level_Sector", "name"])
    .reset_index(drop=True)
)

if "company_id" in df_scrape_ready.columns:
    df_scrape_ready = df_scrape_ready.drop(columns=["company_id"])

df_scrape_ready.insert(
    0,
    "company_id",
    [f"C{str(index + 1).zfill(4)}" for index in range(len(df_scrape_ready))],
)

display(
    df_scrape_ready[
        [
            "company_id",
            "name",
            "country",
            "industry",
            "High_Level_Sector",
            "size",
            "size_bucket",
            "website_url",
            "final_url",
            "domain",
        ]
    ].head()
)

,company_id,name,country,industry,High_Level_Sector,size,size_bucket,website_url,final_url,domain
0,C0001,froneri,australia,food production,"Agriculture, Food & Beverage",10001+,enterprise,https://froneri.com,https://www.froneri.com/,froneri.com
1,C0002,gloria jean’s coffees australia,australia,food & beverages,"Agriculture, Food & Beverage",5001-10000,enterprise,https://gloriajeanscoffees.com.au,https://www.gloriajeans.com.au/,gloriajeanscoffees.com.au
2,C0003,australian subscription television & radio ass...,australia,broadcast media,"Arts, Media & Entertainment",5001-10000,enterprise,https://astra.org.au,https://astra.org.au,astra.org.au
3,C0004,evt - entertainment | ventures | travel,australia,entertainment,"Arts, Media & Entertainment",5001-10000,enterprise,https://evt.com,https://www.evt.com/,evt.com
4,C0005,austal,australia,shipbuilding,"Automotive, Aerospace & Transportation Equipment",5001-10000,enterprise,https://austal.com,https://www.austal.com/,austal.com


## 21. Validate government ratio

This validation proves that government organisations are below or equal to 10%.

It checks both:

- Per country ratio.
- Overall final dataset ratio.

In [ ]:
government_summary = build_government_summary(df_scrape_ready)

print("Government ratio by country:")
display(government_summary)

validate_government_ratio(df_scrape_ready, government_summary)

Government ratio by country:


,country,total_companies,government_companies,government_ratio
0,australia,164,8,0.048780
1,brazil,123,3,0.024390
2,china,109,3,0.027523
3,france,152,5,0.032895
4,germany,172,8,0.046512
5,japan,168,4,0.023810
6,saudi arabia,130,2,0.015385
7,south africa,152,7,0.046053
8,sri lanka,151,7,0.046358
9,turkey,128,2,0.015625


Total government count: 49
Total government ratio: 0.033816425120772944


## 22. Review final balance summaries

These summaries are used for quick quality checking before passing the data to Notebook 2.

In [ ]:
print("--- Country Count ---")
country_count_summary = (
    df_scrape_ready["country"]
    .value_counts()
    .rename_axis("country")
    .reset_index(name="company_count")
)
display(country_count_summary)

print("--- Size Bucket Count ---")
size_count_summary = (
    df_scrape_ready
    .groupby(["country", "size_bucket"])
    .size()
    .reset_index(name="company_count")
    .sort_values(["country", "size_bucket"])
)
display(size_count_summary)

print("--- Sector Count ---")
sector_count_summary = (
    df_scrape_ready
    .groupby(["country", "High_Level_Sector"])
    .size()
    .reset_index(name="company_count")
    .sort_values(["country", "company_count"], ascending=[True, False])
)
display(sector_count_summary)

print("--- Government Ratio Check ---")
display(government_summary)

--- Country Count ---


,country,company_count
0,germany,172
1,japan,168
2,australia,164
3,france,152
4,south africa,152
5,sri lanka,151
6,saudi arabia,130
7,turkey,128
8,brazil,123
9,china,109


--- Size Bucket Count ---


,country,size_bucket,company_count
0,australia,enterprise,38
1,australia,large,42
2,australia,medium,38
3,australia,small,46
4,brazil,enterprise,33
5,brazil,large,35
6,brazil,medium,24
7,brazil,small,31
8,china,enterprise,24
9,china,large,28


--- Sector Count ---


,country,High_Level_Sector,company_count
5,australia,Consumer Goods & Retail,15
4,australia,"Construction, Real Estate & Built Environment",14
13,australia,Information Technology & Internet,13
15,australia,"Logistics, Transportation & Trade",13
11,australia,"Hospitality, Travel & Leisure",12
...,...,...,...
178,turkey,Financial Services & Insurance,5
182,turkey,Industrial Manufacturing & Materials,4
187,turkey,"Nonprofit, Social Services & Religion",4
188,turkey,Other,4


--- Government Ratio Check ---


,country,total_companies,government_companies,government_ratio
0,australia,164,8,0.048780
1,brazil,123,3,0.024390
2,china,109,3,0.027523
3,france,152,5,0.032895
4,germany,172,8,0.046512
5,japan,168,4,0.023810
6,saudi arabia,130,2,0.015385
7,south africa,152,7,0.046053
8,sri lanka,151,7,0.046358
9,turkey,128,2,0.015625


## 23. Save seed sample and CSV outputs

Saved outputs:

- `company_seed_sample.csv`: small testing file for Notebook 2.
- `company_scrape_ready.csv`: production file for Notebook 2.
- `company_duplicate_domain_review.csv`: suspicious duplicate/shared domains to review.
- `company_domain_checked.csv`: full domain matching results.
- `company_reachable_websites.csv`: reachable candidate websites.
- `company_unreachable_websites_reserve.csv`: unreachable candidate websites.
- `government_ratio_summary.csv`: government ratio validation table.

In [ ]:
# One test company per country and size bucket where possible.
df_seed_sample = (
    df_scrape_ready
    .groupby(["country", "size_bucket"], group_keys=False)
    .sample(n=1, random_state=42)
    .reset_index(drop=True)
)

# df_domain_checked.to_csv(OUTPUT_DOMAIN_CHECKED_PATH, index=False)
df_duplicate_domain_review.to_csv(OUTPUT_DUPLICATE_DOMAIN_REVIEW_PATH, index=False)
# df_checked.to_csv(OUTPUT_CLEANED_TARGET_COUNTRIES_PATH, index=False)
# df_reachable.to_csv(OUTPUT_REACHABLE_PATH, index=False)
df_unreachable.to_csv(OUTPUT_UNREACHABLE_PATH, index=False)
government_summary.to_csv(OUTPUT_GOVERNMENT_SUMMARY_PATH, index=False)
df_seed_sample.to_csv(OUTPUT_SEED_SAMPLE_PATH, index=False)
df_scrape_ready.to_csv(OUTPUT_SCRAPE_READY_PATH, index=False)

## 24. Optional Colab download

Run this cell only in Google Colab.

In [ ]:
# Optional: uncomment this block only when running in Google Colab.

# from google.colab import files
# files.download(OUTPUT_SEED_SAMPLE_PATH)
# files.download(OUTPUT_SCRAPE_READY_PATH)
# files.download(OUTPUT_DUPLICATE_DOMAIN_REVIEW_PATH)
# files.download(OUTPUT_GOVERNMENT_SUMMARY_PATH)

In [ ]:
display(
    df_checked["failure_reason"]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

,failure_reason,count
0,None,1449
1,connection_error,413
2,timeout,81
3,http_error: 403,34
4,http_error: 404,10
5,http_error: 500,4
6,unexpected_error: Server disconnected without ...,2
7,http_error: 503,2
8,http_error: 410,1
9,http_error: 401,1
